# РК 1. EDA-анализ данных об SQL-инъекциях

Цель анализа — исследовать структуру преподавательского датасета, распределения категорий и типов атак, используемые инструменты, MITRE-техники, теги, источники и текстовые поля, а также выявить основные закономерности.

Перед запуском выполните `client.py`, чтобы получить `sql_injections_merged.csv`.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

CSV_PATH = Path("sql_injections_merged.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "Файл sql_injections_merged.csv не найден. Сначала выполните: python client.py"
    )

df = pd.read_csv(CSV_PATH)
original_columns = df.columns.tolist()

print(f"Загружено строк: {len(df):,}")
print(f"Столбцов: {len(df.columns)}")
display(df.head())

In [ ]:
def split_values(series, pattern=r"[,;#]", lower=False):
    values = []
    for value in series.dropna().astype(str):
        for part in re.split(pattern, value):
            token = part.strip()
            if lower:
                token = token.lower()
            if token:
                values.append(token)
    return values


def plot_counter(counter, top_n, title):
    top = pd.Series(dict(counter.most_common(top_n)), dtype="int64")
    if top.empty:
        print("Нет данных для построения графика")
        return top

    plt.figure(figsize=(10, max(5, top_n * 0.32)))
    top.sort_values().plot(kind="barh")
    plt.title(title)
    plt.xlabel("Количество")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
    return top


def pct(part, whole):
    return 0.0 if whole == 0 else 100 * part / whole


def fmt_int(value):
    return f"{int(value):,}".replace(",", " ")

## 1. Общая информация: размер, типы, пропуски и дубликаты

In [ ]:
print("Размер DataFrame:", df.shape)
display(df.dtypes.to_frame("dtype"))

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
}).sort_values(["missing_count", "missing_percent"], ascending=False)
display(missing)

plt.figure(figsize=(12, 6))
missing["missing_count"].sort_values().plot(kind="barh")
plt.title("Количество пропусков по столбцам")
plt.xlabel("Количество пропусков")
plt.tight_layout()
plt.show()

duplicates = int(df.duplicated().sum())
duplicate_ids = int(df["id"].duplicated().sum())
total_missing = int(df.isna().sum().sum())
total_cells = int(df.shape[0] * df.shape[1])
max_col = missing.index[0]
max_count = int(missing.iloc[0]["missing_count"])
max_percent = float(missing.iloc[0]["missing_percent"])

print(
    f"Вывод: число строк — {fmt_int(len(df))}, число исходных столбцов — {len(original_columns)}. "
    f"Всего пропущено {total_missing} значений ({pct(total_missing, total_cells):.2f}% всех ячеек); "
    f"максимум пропусков в поле '{max_col}' — {max_count} ({max_percent:.2f}%)."
)
print(
    f"Полных дубликатов строк: {duplicates}, повторяющихся id: {duplicate_ids}. "
    "Следовательно, объём пропусков невелик и структура данных пригодна для дальнейшего EDA без удаления значительной части наблюдений."
)

## 2. Анализ `Category`

In [ ]:
category_counts = df["Category"].fillna("<NULL>").value_counts()
category_share = (df["Category"].fillna("<NULL>").value_counts(normalize=True) * 100).round(2)

category_table = pd.DataFrame({
    "count": category_counts,
    "share_%": category_share,
})
display(category_table.head(10))

plt.figure(figsize=(10, 6))
category_counts.head(10).sort_values().plot(kind="barh")
plt.title("Топ-10 категорий")
plt.xlabel("Количество")
plt.tight_layout()
plt.show()

pie = category_counts.head(8).copy()
other = int(category_counts.iloc[8:].sum())
if other:
    pie.loc["Прочие"] = other

plt.figure(figsize=(8, 8))
plt.pie(pie.values, labels=pie.index, autopct="%1.1f%%", startangle=90)
plt.title("Category: топ-8 + прочие")
plt.tight_layout()
plt.show()

top10_share = pct(category_counts.head(10).sum(), len(df))
print(
    f"Вывод: число уникальных категорий — {df['Category'].nunique(dropna=True)}. "
    f"Наиболее частая — '{category_counts.index[0]}' ({category_counts.iloc[0]} записей; {category_share.iloc[0]:.2f}%)."
)
print(
    f"Топ-10 категорий охватывают {top10_share:.2f}% наблюдений, при этом ни одна категория не занимает подавляющую долю. "
    "Распределение можно считать достаточно равномерным среди ведущих категорий, с выраженным длинным хвостом менее частых значений."
)

## 3. Анализ `Attack Type`

In [ ]:
attack_counts = df["Attack Type"].fillna("<NULL>").value_counts()
display(attack_counts.head(10).to_frame("count"))

plt.figure(figsize=(10, 6))
attack_counts.head(10).sort_values().plot(kind="barh")
plt.title("Топ-10 типов атак")
plt.xlabel("Количество")
plt.tight_layout()
plt.show()

unique_attack_types = df["Attack Type"].nunique(dropna=True)
unique_ratio = pct(unique_attack_types, len(df))
singleton_types = int((df["Attack Type"].value_counts() == 1).sum())
singleton_share = pct(singleton_types, unique_attack_types)
top10_attack_share = pct(attack_counts.head(10).sum(), len(df))

print(
    f"Вывод: наиболее частый тип — '{attack_counts.index[0]}', частота — {attack_counts.iloc[0]}. "
    f"Всего обнаружено {fmt_int(unique_attack_types)} уникальных типов атак, что эквивалентно {unique_ratio:.2f}% от числа строк."
)
print(
    f"{fmt_int(singleton_types)} типов ({singleton_share:.2f}% всех уникальных Attack Type) встречаются только один раз, "
    f"а топ-10 типов покрывают лишь {top10_attack_share:.2f}% датасета. Это указывает на очень высокую уникальность и разнообразие поля `Attack Type`."
)

## 4. Анализ `Tools Used`

In [ ]:
tools = split_values(df["Tools Used"], pattern=r"[,;#]")
tools_counter = Counter(tools)
top_tools = plot_counter(tools_counter, 15, "Топ-15 инструментов")
display(top_tools.sort_values(ascending=False).to_frame("count"))

top5_tools = tools_counter.most_common(5)
print(
    "Вывод: наиболее часто встречаются "
    + ", ".join(f"{name} ({count})" for name, count in top5_tools)
    + "."
)
print(
    "Набор лидирующих инструментов сочетает средства веб-тестирования, анализа сетевого трафика и скриптовой/CLI-автоматизации. "
    "Это показывает, что типичный технический стек в датасете не ограничивается одним классом инструментов."
)

## 5. Анализ `MITRE Technique`

In [ ]:
mitre_pattern = r"T\d{4}(?:\.\d{3})?"
mitre_all = []

for value in df["MITRE Technique"].dropna().astype(str):
    mitre_all.extend(re.findall(mitre_pattern, value))

mitre_counter = Counter(mitre_all)
top_mitre = plot_counter(mitre_counter, 10, "Топ-10 MITRE ATT&CK техник")
display(top_mitre.sort_values(ascending=False).to_frame("count"))

top5_mitre = mitre_counter.most_common(5)
print(
    "Вывод: наиболее частые MITRE-техники — "
    + ", ".join(f"{name} ({count})" for name, count in top5_mitre)
    + "."
)
print(
    f"Всего из поля `MITRE Technique` извлечено {fmt_int(len(mitre_all))} идентификаторов, соответствующих заданному шаблону MITRE-кода. "
    "Разнообразие техник согласуется с высокой вариативностью типов атак; их непосредственная связь с `Attack Type` отдельно рассматривается в пункте 9."
)

## 6. Анализ `Tags`

In [ ]:
tags = split_values(df["Tags"], pattern=r"[,;#]", lower=True)
tags_counter = Counter(tags)
top_tags = plot_counter(tags_counter, 20, "Топ-20 тегов")
display(top_tags.sort_values(ascending=False).to_frame("count"))

top5_tags = tags_counter.most_common(5)
print(
    "Вывод: наиболее частые теги — "
    + ", ".join(f"{name} ({count})" for name, count in top5_tags)
    + "."
)
print(
    "Лидирующие теги `redteam`, `phishing`, `mitm`, `persistence` и `android` показывают, "
    "что в наборе заметно представлены как наступательные сценарии, так и сетевые, социальные и платформенные аспекты атак."
)

## 7. Анализ `Source`

In [ ]:
sources = split_values(df["Source"], pattern=r"[,;]")
sources_counter = Counter(sources)
top_sources = plot_counter(sources_counter, 15, "Топ-15 источников")
display(top_sources.sort_values(ascending=False).to_frame("count"))

top5_sources = sources_counter.most_common(5)
print(
    "Вывод: наиболее часто встречаются источники/обозначения "
    + ", ".join(f"{name} ({count})" for name, count in top5_sources)
    + "."
)
print(
    "По частоте лидирует `Simulated`. Среди явно именованных отраслевых источников особенно заметны OWASP, MITRE и MITRE ATT&CK, "
    "которые формируют существенную часть ссылочной базы датасета."
)

## 8. `Category × Attack Type`

In [ ]:
top_categories = df["Category"].value_counts().head(10).index
top_attacks = df["Attack Type"].value_counts().head(10).index

subset = df[
    df["Category"].isin(top_categories)
    & df["Attack Type"].isin(top_attacks)
]

ct_cat_attack = pd.crosstab(subset["Category"], subset["Attack Type"])
display(ct_cat_attack)

plt.figure(figsize=(14, 8))
sns.heatmap(ct_cat_attack, annot=True, fmt="d", cmap="Blues")
plt.title("Category × Attack Type: абсолютные значения")
plt.tight_layout()
plt.show()

row_share = ct_cat_attack.div(ct_cat_attack.sum(axis=1).replace(0, np.nan), axis=0)

plt.figure(figsize=(14, 8))
sns.heatmap(row_share, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Category × Attack Type: доли по строке")
plt.tight_layout()
plt.show()

if not ct_cat_attack.empty:
    strongest = ct_cat_attack.stack().sort_values(ascending=False)
    pair = strongest.index[0]
    count = int(strongest.iloc[0])
    pair_share = float(row_share.loc[pair[0], pair[1]])

    print(
        f"Вывод: наиболее выраженная пара — '{pair[0]}' → '{pair[1]}', частота — {count}. "
        f"В строке этой категории внутри матрицы топ-10 данный тип составляет {pair_share:.2%}."
    )
    print(
        "Следовательно, между отдельными категориями и типами атак присутствуют выраженные частотные связи; "
        "абсолютная и нормированная heatmap позволяют увидеть их с разных сторон."
    )

## 9. `Attack Type × MITRE`

In [ ]:
def first_mitre(value):
    if pd.isna(value):
        return np.nan
    found = re.findall(mitre_pattern, str(value))
    return found[0] if found else np.nan


df["mitre_main"] = df["MITRE Technique"].apply(first_mitre)

top8_attack = df["Attack Type"].value_counts().head(8).index
top10_mitre = df["mitre_main"].value_counts().head(10).index

subset = df[
    df["Attack Type"].isin(top8_attack)
    & df["mitre_main"].isin(top10_mitre)
]

ct_attack_mitre = pd.crosstab(subset["Attack Type"], subset["mitre_main"])
display(ct_attack_mitre)

plt.figure(figsize=(14, 7))
sns.heatmap(ct_attack_mitre, annot=True, fmt="d", cmap="Purples")
plt.title("Attack Type × MITRE Technique")
plt.tight_layout()
plt.show()

if not ct_attack_mitre.empty:
    strongest = ct_attack_mitre.stack().sort_values(ascending=False)
    pair = strongest.index[0]
    count = int(strongest.iloc[0])

    print(
        f"Вывод: наиболее выраженная связь в выбранной матрице — '{pair[0]}' с техникой {pair[1]} "
        f"({count} записей)."
    )
    print(
        "Тепловая карта показывает, что одни и те же MITRE-техники могут встречаться у разных `Attack Type`, "
        "но для отдельных типов атак формируются заметные локальные максимумы."
    )

## 10. Длины текстовых полей

In [ ]:
text_columns = {
    "Scenario Description": "len_scenario_description",
    "Attack Steps": "len_attack_steps",
    "Solution": "len_solution",
}

for source_col, length_col in text_columns.items():
    df[length_col] = df[source_col].fillna("").astype(str).str.len()

length_cols = list(text_columns.values())
display(df[length_cols].describe().round(1))

outlier_rows = []
for source_col, length_col in text_columns.items():
    values = df[length_col]
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = int(((values < lower) | (values > upper)).sum())
    outlier_rows.append({
        "field": source_col,
        "mean": round(values.mean(), 1),
        "median": round(values.median(), 1),
        "max": int(values.max()),
        "iqr_outliers": outliers,
    })

outlier_table = pd.DataFrame(outlier_rows).set_index("field")
display(outlier_table)

for source_col, length_col in text_columns.items():
    plt.figure(figsize=(10, 5))
    sns.histplot(x=df[length_col], bins=40, kde=True, color="steelblue")
    plt.title(f"Распределение длины: {source_col}")
    plt.xlabel("Количество символов")
    plt.tight_layout()
    plt.show()

means = df[length_cols].mean().sort_values(ascending=False)
longest = means.index[0]
source_longest = {v: k for k, v in text_columns.items()}[longest]

print(
    f"Вывод: самым длинным текстовым полем является `Attack Steps`: средняя длина {df['len_attack_steps'].mean():.1f}, "
    f"медиана {df['len_attack_steps'].median():.1f}, максимум {fmt_int(df['len_attack_steps'].max())} символов."
)
print(
    "Выбросы по правилу 1.5×IQR присутствуют во всех трёх распределениях; это отражает наличие записей "
    "с существенно более подробными описаниями, шагами атаки или рекомендациями по защите."
)

## 11. Частые слова в `Solution`

In [ ]:
STOP_WORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "with", "by", "from",
    "is", "are", "be", "as", "at", "that", "this", "these", "those", "it", "its", "into",
    "can", "should", "would", "could", "such", "using", "use", "used", "ensure", "all",
    "your", "their", "they", "you", "we", "not", "if", "when", "where", "which", "who",
    "application", "applications", "system", "systems", "data"
}

solution_words = []
for text in df["Solution"].dropna().astype(str):
    tokens = re.findall(r"[A-Za-z][A-Za-z\-]+", text.lower())
    solution_words.extend(
        token for token in tokens
        if token not in STOP_WORDS and len(token) > 2
    )

solution_counter = Counter(solution_words)
top_solution = plot_counter(solution_counter, 20, "Топ-20 слов в Solution")
display(top_solution.sort_values(ascending=False).to_frame("count"))

top10_solution = solution_counter.most_common(10)
print(
    "Вывод: наиболее частая лексика рекомендаций — "
    + ", ".join(f"{name} ({count})" for name, count in top10_solution)
    + "."
)
print(
    "По смыслу наиболее частые рекомендации сосредоточены вокруг ограничения доступа, блокирования/отключения опасных возможностей, "
    "усиления защиты, проверки и валидации входных данных, а также мониторинга."
)

## 12. Финальная сводка по исходным полям

In [ ]:
summary = pd.DataFrame({
    "dtype": df[original_columns].dtypes.astype(str),
    "non_null": df[original_columns].notna().sum(),
    "missing": df[original_columns].isna().sum(),
    "missing_%": (df[original_columns].isna().mean() * 100).round(2),
    "unique": df[original_columns].nunique(dropna=True),
})

top_values = []
top_counts = []

for column in original_columns:
    vc = df[column].dropna().astype(str).value_counts()
    if len(vc):
        top_values.append(vc.index[0])
        top_counts.append(int(vc.iloc[0]))
    else:
        top_values.append(np.nan)
        top_counts.append(0)

summary["top_value"] = top_values
summary["top_count"] = top_counts

display(summary)
print(
    f"Вывод: максимальная уникальность наблюдается в полях, описывающих конкретные сценарии и типы атак; "
    f"`Attack Type` содержит {fmt_int(df['Attack Type'].nunique(dropna=True))} уникальных значений. "
    "При этом доля пропусков по каждому отдельному столбцу невелика."
)

## Финальные выводы

In [ ]:
main_categories = df["Category"].value_counts().head(3)
main_attacks = df["Attack Type"].value_counts().head(3)

print("1. Категории и типы атак")
print(
    "   Ведущие категории: "
    + ", ".join(f"{name} ({count})" for name, count in main_categories.items())
    + "."
)
print(
    "   Ведущие типы атак: "
    + ", ".join(f"{name} ({count})" for name, count in main_attacks.items())
    + f". При этом поле Attack Type характеризуется высокой вариативностью: "
      f"{fmt_int(df['Attack Type'].nunique())} уникальных значений."
)

print("\n2. Инструменты")
print(
    "   Наиболее популярны "
    + ", ".join(f"{name} ({count})" for name, count in tools_counter.most_common(5))
    + ". Их сочетание отражает использование веб-тестирования, анализа трафика и автоматизации."
)

print("\n3. MITRE-техники")
print(
    "   Наиболее часто встречаются "
    + ", ".join(f"{name} ({count})" for name, count in mitre_counter.most_common(5))
    + ". Эти техники образуют основное ядро MITRE-разметки по частоте."
)

print("\n4. Теги и источники")
print(
    "   Частые теги: "
    + ", ".join(f"{name} ({count})" for name, count in tags_counter.most_common(5))
    + "."
)
print(
    "   По источникам лидирует Simulated; среди именованных отраслевых источников "
    "особенно заметны OWASP, MITRE и MITRE ATT&CK."
)

print("\n5. Рекомендации по защите")
print(
    "   Наиболее частая лексика Solution: "
    + ", ".join(name for name, _ in solution_counter.most_common(10))
    + ". В целом рекомендации сосредоточены на ограничении доступа, блокировании опасных возможностей, "
      "валидации входных данных и мониторинге."
)

total_missing = int(df[original_columns].isna().sum().sum())
total_cells = int(df[original_columns].shape[0] * df[original_columns].shape[1])

print("\n6. Качество данных")
print(
    f"   Пропущено {total_missing} значений ({pct(total_missing, total_cells):.2f}% всех исходных ячеек); "
    f"полных дубликатов — {int(df[original_columns].duplicated().sum())}. "
    "Низкая доля пропусков не оказывает существенного влияния на основные результаты EDA."
)

print("\n7. Общий вывод")
print(
    f"   Датасет содержит {fmt_int(len(df))} наблюдений, {df['Category'].nunique()} категории и "
    f"{fmt_int(df['Attack Type'].nunique())} уникальных типов атак, а также разнообразные инструменты, "
    "MITRE-техники, теги и источники. По объёму, полноте и широте представленных признаков "
    "набор можно считать репрезентативным для учебного изучения SQL-инъекций в рамках поставленной задачи: "
    "он позволяет проводить полноценный EDA, выявлять устойчивые частотные закономерности "
    "и анализировать связи между признаками."
)